In [1]:
import os

In [2]:
%pwd

'c:\\Users\\HP\\Documents\\LLM_learning\\End-to-end-ML-with-mlflow\\research'

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir : Path

In [5]:
from ml_project.constants import *
from ml_project.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_file_path=CONFIG_FILE_PATH, 
        params_file_path=PARAMS_FILE_PATH, 
        schema_file_path=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )
        return data_ingestion_config

In [7]:
import urllib.request as request
import zipfile  
from ml_project import logger
from ml_project.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            
            filename,headers=request.urlretrieve(url=self.config.source_URL, filename=self.config.local_data_file)
            logger.info(f"{filename} download! with following info : \n{headers}")
        else:
            logger.info(f"File already exists of size : {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):

        unzip_path = self.config.unzip_dir
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-12 17:08:15,054]: INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-03-12 17:08:15,060]: INFO:common:yaml file: params.yaml loaded successfully]
[2026-03-12 17:08:15,066]: INFO:common:yaml file: schema.yaml loaded successfully]
[2026-03-12 17:08:15,069]: INFO:common:created directory at: artifacts]
[2026-03-12 17:08:15,072]: INFO:common:created directory at: artifacts/data_ingestion]
[2026-03-12 17:08:16,074]: INFO:3371646267:artifacts/data_ingestion/data.zip download! with following info : 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: F782:0A03:C55FC:1F6ABA:69B2A5A7
Accept-Ranges: bytes
Date: Thu, 12 Mar 202